# VascuQuest JAX split-solver qualification

This notebook qualifies the structure-preserving accelerated Virtual Disease solver on **one frozen canonical PWDB subject across all four disease models**. It is numerical/software qualification, not clinical validation.

The accelerated scheme is `jax-exact-loss-rkc2-voigt-ssprk2-v1`. Durable PASS evidence for all four accelerated disease solves from revision `19c6a24d5ec5...` may be reused only after this notebook proves that the numerical split solver, frozen NumPy reference operator, network discretisation and disease physics have not changed. Current NumPy/JAX operator-equivalence gates are always re-executed.

The old full-period explicit NumPy anchor is not a release gate: real-PWDB Voigt stiffness requires millions of explicit steps per cardiac cycle. Instead, numerical identity is tested directly at the frozen semidiscrete NumPy operator/stability level and the accelerated integrator must freshly demonstrate temporal self-convergence over one complete cardiac cycle. The final report cannot become `PASS` until that temporal gate passes.

A fail-fast preflight checks for orphaned qualification processes, GPU occupancy, heavyweight import timing, JAX device discovery and a synchronized tiny JIT before numerical execution.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess, sys, json

REPO = Path('/content/VascuQuest-jax-split-qualification')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch',
    'release/parameterized-cohort-qualification',
    'https://github.com/KNOWDYN/VascuQuest.git', str(REPO)
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
CODE_REVISION = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Code revision:', CODE_REVISION, flush=True)


In [ ]:
# Stage canonical PWDB artifacts to local SSD. No recursive Drive search.
DRIVE_PWDB_SOURCE = Path('/content/drive/MyDrive/VQ_WallWork_CBM/source/PWDB_3275625')
LOCAL_SOURCE = Path('/content/vascuquest-pwdb-source')
OUTPUT_ROOT = Path('/content/drive/MyDrive/VascuQuest/jax_split_one_subject_qualification') / CODE_REVISION[:12]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
STAGE_REPORT = OUTPUT_ROOT / 'source-stage.json'
REPORT = OUTPUT_ROOT / 'jax-split-one-subject-qualification.json'
TRUSTED_REUSE_REVISION = '19c6a24d5ec571946440927344801d3a0a40e78d'
TRUSTED_REUSE_REPORT = Path('/content/drive/MyDrive/VascuQuest/jax_split_one_subject_qualification/19c6a24d5ec5/jax-split-one-subject-qualification.json')

# Reuse is allowed only when actual numerical/scientific implementation files are unchanged.
NUMERICAL_PATHS = [
    'src/vascuquest/disease/solver/jax_split_disease.py',
    'src/vascuquest/disease/solver/jax_disease.py',
    'src/vascuquest/disease/solver/disease_finite_volume.py',
    'src/vascuquest/disease/solver/network.py',
    'src/vascuquest/disease/solver/boundaries.py',
    'src/vascuquest/disease/solver/losses.py',
    'src/vascuquest/disease/physics',
]
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', TRUSTED_REUSE_REVISION], check=True)
changed = subprocess.check_output([
    'git', '-C', str(REPO), 'diff', '--name-only', TRUSTED_REUSE_REVISION, CODE_REVISION, '--', *NUMERICAL_PATHS
], text=True).strip().splitlines()
if changed:
    raise RuntimeError('Trusted JAX evidence cannot be reused because numerical/scientific files changed: ' + ', '.join(changed))
print('Trusted-evidence numerical lineage gate: PASS', flush=True)

stage_cmd = [
    sys.executable, '-u', str(REPO / 'tests/full_data/parameterized_cohort_colab_stage.py'),
    '--drive-source-dir', str(DRIVE_PWDB_SOURCE),
    '--local-source', str(LOCAL_SOURCE),
    '--report', str(STAGE_REPORT),
]
print('$', ' '.join(stage_cmd), flush=True)
subprocess.run(stage_cmd, check=True, cwd=REPO)
print(STAGE_REPORT.read_text())
print('PWDB local-SSD source gate: PASS', flush=True)
print('Trusted four-disease evidence:', TRUSTED_REUSE_REPORT if TRUSTED_REUSE_REPORT.exists() else '<none>', flush=True)


In [ ]:
# Fail-fast environment/import/device preflight. This must finish quickly.
preflight_cmd = [sys.executable, '-u', str(REPO / 'tests/full_data/jax_split_colab_preflight.py')]
print('$', ' '.join(preflight_cmd), flush=True)
try:
    subprocess.run(preflight_cmd, check=True, cwd=REPO, timeout=180)
except subprocess.TimeoutExpired as exc:
    raise RuntimeError('JAX qualification preflight exceeded 180 seconds. Reset/delete the Colab runtime; do not launch the numerical qualification.') from exc
print('Qualification preflight: PASS', flush=True)


In [ ]:
# Confirm the package surface before spending GPU time.
from vascuquest.disease.solver import create_disease_solver
from vascuquest.disease.solver.disease_finite_volume import DiseaseOneDSolver
from vascuquest.disease.solver.jax_split_disease import JAX_SPLIT_SCHEME_ID, JaxDiseaseOneDSolver
numpy_solver = create_disease_solver()
accelerated_solver = create_disease_solver(backend='jax')
assert isinstance(numpy_solver, DiseaseOneDSolver)
assert isinstance(accelerated_solver, JaxDiseaseOneDSolver)
print('NumPy frozen default: PASS')
print('Accelerated scheme:', JAX_SPLIT_SCHEME_ID)


In [ ]:
# Revalidate the current operators and retain the already-qualified full disease solves.
cmd = [
    sys.executable, '-u', str(REPO / 'tests/full_data/jax_split_one_subject_qualification.py'),
    '--source', str(LOCAL_SOURCE),
    '--report', str(REPORT),
    '--code-revision', CODE_REVISION,
]
if TRUSTED_REUSE_REPORT.exists():
    cmd += ['--reuse-report', str(TRUSTED_REUSE_REPORT)]
print('$', ' '.join(cmd), flush=True)
completed = subprocess.run(cmd, cwd=REPO)
if completed.returncode != 0:
    if REPORT.exists():
        print('Persisted failure/partial record:')
        print(REPORT.read_text())
    raise RuntimeError(f'JAX split main qualification failed with exit code {completed.returncode}')
main_payload = json.loads(REPORT.read_text())
assert main_payload['status'] == 'AWAITING_TEMPORAL_REFINEMENT'
assert main_payload.get('main_stage', {}).get('status') == 'PASS'
print('Four-disease/operator main stage: PASS; temporal refinement pending')


In [ ]:
# Require approximately second-order temporal self-convergence over one complete cycle.
refine_cmd = [
    sys.executable, '-u', str(REPO / 'tests/full_data/jax_split_temporal_refinement.py'),
    '--source', str(LOCAL_SOURCE),
    '--report', str(REPORT),
]
print('$', ' '.join(refine_cmd), flush=True)
refined = subprocess.run(refine_cmd, cwd=REPO)
if refined.returncode != 0:
    if REPORT.exists():
        print('Qualification record after temporal-refinement failure:')
        print(REPORT.read_text())
    raise RuntimeError(f'Temporal-refinement qualification failed with exit code {refined.returncode}')
print('Temporal-order gate: PASS')


In [ ]:
payload = json.loads(REPORT.read_text())
assert payload['status'] == 'PASS'
assert payload.get('qualification_complete') is True
assert payload.get('temporal_refinement', {}).get('status') == 'PASS'
print(json.dumps({
    'status': payload['status'],
    'qualification_complete': payload['qualification_complete'],
    'code_revision': payload['code_revision'],
    'canonical_subject_id': payload['canonical_subject_id'],
    'source_age_years': payload['source_age_years'],
    'numerical_scheme_id': payload['numerical_scheme_id'],
    'evidence_reuse': payload.get('evidence_reuse'),
    'reference_strategy': payload['reference_strategy'],
    'main_stage': payload['main_stage'],
    'temporal_refinement': payload['temporal_refinement'],
}, indent=2, sort_keys=True))
print('\nPer-disease limiter attribution:')
for case in payload['cases']:
    info = case['accelerated_full_solve']['limiter_attribution']
    reused = case.get('reuse_provenance', {}).get('reused', False)
    print(case['condition'], 'REUSED' if reused else 'FRESH', json.dumps(info, sort_keys=True))
print('\nDurable evidence:', REPORT)
